## Merge availalbe swaths (F1, F2, F3) into a single merged directory
To be run in `gmtsar` env. 
This is a notebook I developed while trying to figure out how to deal with some subset of F* directories, instead of all the expected ones.
It is still useful, but the merge problem has been solved by modifications to the `p2p_S1_TOPS_Frame.csh` script. Try that first, then this can be used to guide any custom merging needed. 

In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np
import os
import shutil
from pathlib import Path


# pairscsv = '/Volumes/T9_InSAR/2026-04-14_nevada/S1_D144_gmtsar_stack/raw/pairs.csv'
basedir = '/Volumes/T9_InSAR/2026-04-14_nevada/S1_A64/' # Location of F1, F2, F3 directories

# read in pairs information writing from `safe2pairs.ipynb`
pairs_df = pd.read_csv(basedir  + 'raw/pairs.csv')
pairs_df['reference_datetime'] = pd.to_datetime(pairs_df['reference_datetime'])
pairs_df['repeat_datetime'] = pd.to_datetime(pairs_df['repeat_datetime'])



In [6]:
# doypairs=['2026098_2026110','2026098_2026116','2026104_2026110','2026104_2026116','2026110_2026116']

# doypairs = ['2026092_2026104']
doypairs = ['2026111_2026135']

for doypair in doypairs:
    print(doypair)
    # doypair = '2026098_2026110'
    ref_doy_search, rep_doy_search = doypair.split('_')
    print(ref_doy_search, rep_doy_search)
    # search for the index of the row with pairs_df['ref_doy'] == ref_doy_search
    type(pairs_df['ref_doy'][0])
    pairs_df_search = pairs_df[(pairs_df['ref_doy'] == int(ref_doy_search)) & (pairs_df['rep_doy'] == int(rep_doy_search))]
    index = pairs_df_search.index[0]

    # Merge availalbe swaths (F1, F2, F3) into a single merged directory
    ##############################
    #       Set variables       #
    ##############################


    # Convert from datetime object to day of year (DOY), GMTSAR seems to be off by one day. 
    ref_doy = int(pairs_df['reference_datetime'][index].strftime("%j"))-1
    rep_doy = int(pairs_df['repeat_datetime'][index].strftime("%j"))-1

    # Convert from datetime object to YYYYMMDD
    ref_date = pairs_df['reference_datetime'][index].strftime("%Y%m%d")
    rep_date = pairs_df['repeat_datetime'][index].strftime("%Y%m%d")

    # return a list of indices for a given category 
    integer_positions = np.where(pairs_df['category'] == 'coseismic')[0]

    refdoy_str = str(pairs_df['reference_datetime'][3].strftime("%Y")) + f"{ref_doy:03}"
    repdoy_str = str(pairs_df['repeat_datetime'][3].strftime("%Y")) + f"{rep_doy:03}"
    print(f"Working on date pair: {refdoy_str}_{repdoy_str}")

    grdtypes = ["xphase","yphase",'phasefilt','phase','corr','mask','los','unwrap']    # ,

    # Check which swaths are present: 
    swaths_with_file = []
    for swath in ['F1', 'F2', 'F3']:
        transdat_file = Path(f"{basedir}/{swath}/topo/trans.dat")
        if transdat_file.is_file():
            swaths_with_file.append(swath)
    print(swaths_with_file)

    # Make the merged directory if it doesn't already exist
    merge_dir = f'{basedir}/merge_{ref_date}_{rep_date}'  # Output directory for merged results
    os.makedirs(merge_dir, exist_ok=True)
    shutil.copy2(f"{basedir}/{swaths_with_file[0]}/topo/trans.dat", f"{merge_dir}")
    # Symlink S1_20260404_015053_F1.LED file (from F1/raw) to the merged directory
    rawdir = Path(f'{basedir}/F2/raw/')
    ledpattern = f"S1_{ref_date}*.LED"
    prm_ref_file = next(
        rawdir.glob(ledpattern),
        None
    )
    shutil.copy2(prm_ref_file, f"{merge_dir}")

    # Copy or symlink the filter file (e.g., F1/intf/2026105_2026117/gauss_100) to the merge directory 
    basedir = Path(f"{basedir}")
    filtpattern = f"*/intf/{refdoy_str}_{repdoy_str}/gauss*"
    filt_file = next(
        basedir.glob(filtpattern),
        None
    )
    print(filt_file)
    shutil.copy2(filt_file, f"{merge_dir}")

    ##################################################
    #       Loop through each grd type & swath       #
    ##################################################

    ## Loop through each grd type (xphase, yphase, phasefilt)
    for i, grdtype in enumerate(grdtypes):
        # check if the relevant grd file type exists: 
        grd_file = Path(f"{basedir}/{swaths_with_file[0]}/intf/{refdoy_str}_{repdoy_str}/{grdtype}.grd")
        output_lines = []
        if Path(grd_file).is_file():
            print(f"Found {grdtype} file: {grd_file}")
            # Loop through each swath 
            for swath in swaths_with_file:
                print(f"Working on swath {swath}")
                slcdir = Path(f"{basedir}/{swath}/slc/")
                intfdir = Path(f"{basedir}/{swath}/intf/{refdoy_str}_{repdoy_str}/")
                if intfdir.is_dir():
                    pattern = f"S1_{ref_date}*.PRM"
                    prm_ref_file = next(
                        intfdir.glob(pattern),
                        None
                    )
                    grd_file = f"{intfdir}/{grdtype}.grd"
                    
                    if prm_ref_file:    # Check if the PRM file exists, if so, assume that the corresponding GRD file also exists. 
                        print(prm_ref_file) 
                        # Find the grd file you want to merge
                        output_lines.append(str(prm_ref_file)+':'+str(grd_file)+'\n')  # Append the file name to the 
                        print(grd_file)
                        # Check if grdtype exists

                    else:
                        print(f"No file matching: {pattern}")

                else:
                    print(f"No {swath} dir found")
                ## Write the output text file
                output_file = Path(f"{basedir}/merge_{ref_date}_{rep_date}/inputlist_{grdtype}.txt")
        else:
            print(f"No file matching: {grd_file}")
            merge_dir = f'{basedir}/merge_{ref_date}_{rep_date}'  # Output directory for merged results
        

        if output_lines:
            output_file = Path(
                f"{basedir}/merge_{ref_date}_{rep_date}/inputlist_{grdtype}.txt"
            )

            with open(output_file, "w") as f:
                f.writelines(line + "\n" for line in output_lines)

            print(f"Wrote {len(output_lines)} lines to {output_file}")



2026111_2026135
2026111 2026135
Working on date pair: 2026111_2026135
['F1', 'F2']
/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F1/intf/2026111_2026135/gauss_100
Found xphase file: /Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F1/intf/2026111_2026135/xphase.grd
Working on swath F1
/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F1/intf/2026111_2026135/S1_20260422_015152_F1.PRM
/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F1/intf/2026111_2026135/xphase.grd
Working on swath F2
/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F2/intf/2026111_2026135/S1_20260422_015153_F2.PRM
/Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F2/intf/2026111_2026135/xphase.grd
Wrote 2 lines to /Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/merge_20260422_20260516/inputlist_xphase.txt
Found yphase file: /Volumes/T9_InSAR/2026-04-14_nevada/S1_A064_gmtsar_stack/F1/intf/2026111_2026135/yphase.grd
Working on swath F1
/Volumes/T9_InSAR/2026-04-1

## Run merge and geocode script from the command line
`/Volumes/T9_InSAR/2026-04-14_nevada/S1_D144_gmtsar_stack/merge_geocode.sh`